## 這份是晚上無人的時候用來下載 FinMind數據的

In [1]:
from src.config import RAM_PATH
from src.factor_engine.engine import FactorEngine
if globals().get("engine", None) is None:
    engine = FactorEngine(
        qlib_data_path=RAM_PATH,
        instrument="all",
        start_time='2019-01-01',
        end_time='2026-03-06',
        device="cpu",               # 支援 "cpu" / "cuda:0" 字串或 torch.device
        max_backtrack_days=0,
        max_future_days=0,
        adjust_factor = 'back'
    )
    df_tradable = engine.load_raw_df("$is_tradable")
    tej_df = engine.load_raw_df(['$open' , '$high' , '$low' , '$close' , '$volume' , '$is_tradable'])

data path: /dev/shm/qlib_ram


[248387:MainThread](2026-03-08 11:40:26,392) INFO - qlib.Initialization - [config.py:452] - default_conf: client.
[248387:MainThread](2026-03-08 11:40:26,900) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.
[248387:MainThread](2026-03-08 11:40:26,901) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/dev/shm/qlib_ram')}


[FactorEngine] Loading data: instrument=all, period=[2019-01-01, 2026-03-06], device=cpu
[FactorEngine] Data loaded: 1739 days, 2013 stocks, 10 features, tensor shape=(1739, 10, 2013)


In [2]:
tej_df.reset_index(inplace=True)
tej_df.head()

,datetime,instrument,$open,$high,$low,$close,$volume,$is_tradable
0,2019-01-02,1101,35.750000,35.950001,35.500000,35.500000,6566.0,1.0
1,2019-01-02,1102,34.400002,34.400002,33.599998,33.799999,3677.0,1.0
2,2019-01-02,1103,13.650000,13.650000,13.500000,13.600000,248.0,1.0
3,2019-01-02,1104,18.900000,18.900000,18.650000,18.900000,192.0,1.0
4,2019-01-02,1108,7.300000,7.410000,7.300000,7.410000,43.0,1.0


In [3]:
unique_dates = tej_df['datetime'].unique()
unique_dates = [date.strftime("%Y-%m-%d") for date in unique_dates]
print(unique_dates[:10])

['2019-01-02', '2019-01-03', '2019-01-04', '2019-01-07', '2019-01-08', '2019-01-09', '2019-01-10', '2019-01-11', '2019-01-14', '2019-01-15']


In [4]:
from src.config import FINMIND_TOKEN
from FinMind.data import DataLoader
from loguru import logger
import sys
logger.remove()  # 移除預設 handler
logger.add(sys.stderr, level="WARNING")
finmind_api = DataLoader()
finmind_api.login_by_token(api_token=FINMIND_TOKEN)

True

In [5]:
import time
from datetime import datetime
print("當前時間:", datetime.now())
now = datetime.now()
is_before_8am = now.hour > 8
print("現在是8點之前嗎？", is_before_8am)

當前時間: 2026-03-08 11:40:36.334915
現在是8點之前嗎？ True


In [6]:
skip_date = '2019-02-20'
skip_date in unique_dates

True

In [ ]:
from src.utils.RateLimiter import RateLimiter, retry_on_not_ready
from tqdm import tqdm
import pandas as pd 
import json
import os 
rate_limiter_finmind = RateLimiter(max_requests=98, time_window=60)
@retry_on_not_ready(max_retries=3, delay=2, rate_limiter=rate_limiter_finmind)
def fetch_data_finmind(api, code, date):
    """帶重試機制和速率限制的 kbars 請求"""    
    return api.taiwan_stock_kbar(
        stock_id=code,
        date=date
    )

In [8]:
import telebot 
from src.config import TELEGRAM_BOT_TOKEN , TELEGRAM_CHAT_ID
bot = telebot.TeleBot(TELEGRAM_BOT_TOKEN)

In [ ]:
save_path = '/home/newt/quant_project/1min_data'
for target_date in unique_dates:
    ohlcv_path = os.path.join(save_path, f'{target_date}.parquet')
    if os.path.exists(ohlcv_path):
        continue
    if target_date == skip_date:
        continue
    code_list = tej_df[tej_df['datetime']==target_date]['instrument'].unique().tolist()
    fail_task = []
    result_df_list =[]
    no_trade_record = []
    start = datetime.now()

    if len(code_list) > 0 :
        with tqdm(total=len(code_list), desc=f"日期: {target_date}", unit="筆") as pbar:
            while len(code_list) > 0 :
                code = code_list.pop(0)
                try:
                    df = fetch_data_finmind(
                    finmind_api, 
                    code, 
                    target_date,
                    )
                    if df.empty:
                        no_trade_record.append(code)
                    result_df_list.append(df)
                except Exception as e :
                        fail_task.append(code)
                finally :
                    pbar.update(1)
    df_result = pd.concat(result_df_list)
    
    df_result.to_parquet(ohlcv_path)
    
    # 將 result_df_list 存成 JSON
    fail_task_path = os.path.join(save_path, f'{target_date}_fail.json')
    with open(fail_task_path, 'w', encoding='utf-8') as f:
        json.dump(fail_task, f, ensure_ascii=False, indent=2)

    # 將 no_trade_record 存成 JSON
    no_trade_json_path = os.path.join(save_path, f'{target_date}_no_trade_record.json')
    with open(no_trade_json_path, 'w', encoding='utf-8') as f:
        json.dump(no_trade_record, f, ensure_ascii=False, indent=2)
    if datetime.now().hour > 8:
        bot.send_message(chat_id=TELEGRAM_CHAT_ID, text='即將開始下一輪 等候10秒')
        time.sleep(10)
    

日期: 2019-01-03:   0%|          | 0/1685 [00:00<?, ?筆/s]

日期: 2019-01-08: 100%|██████████| 1685/1685 [21:33<00:00,  1.30筆/s]  


KeyboardInterrupt: 

In [ ]:
save_path = '/home/newt/quant_project/1min_data'
target_date = '2026-03-06'
ohlcv_path = os.path.join(save_path, f'{target_date}.parquet')
if os.path.exists(ohlcv_path):
    continue
if target_date == skip_date:
    continue
code_list = tej_df[tej_df['datetime']==target_date]['instrument'].unique().tolist()
fail_task = []
result_df_list =[]
no_trade_record = []
start = datetime.now()

if len(code_list) > 0 :
    with tqdm(total=len(code_list), desc=f"日期: {target_date}", unit="筆") as pbar:
        while len(code_list) > 0 :
            code = code_list.pop(0)
            try:
                df = fetch_data_finmind(
                finmind_api, 
                code, 
                target_date,
                )
                if df.empty:
                    no_trade_record.append(code)
                result_df_list.append(df)
            except Exception as e :
                    print(e)
            finally :
                pbar.update(1)
df_result = pd.concat(result_df_list)

df_result.to_parquet(ohlcv_path)

# 將 result_df_list 存成 JSON
fail_task_path = os.path.join(save_path, f'{target_date}_fail.json')
with open(fail_task_path, 'w', encoding='utf-8') as f:
    json.dump(fail_task, f, ensure_ascii=False, indent=2)

# 將 no_trade_record 存成 JSON
no_trade_json_path = os.path.join(save_path, f'{target_date}_no_trade_record.json')
with open(no_trade_json_path, 'w', encoding='utf-8') as f:
    json.dump(no_trade_record, f, ensure_ascii=False, indent=2)
if datetime.now().hour > 8:
    bot.send_message(chat_id=TELEGRAM_CHAT_ID, text='即將開始下一輪 等候10秒')
    time.sleep(10)
    